# Blackbody wavelength spectrum

## Prerequisites

Review thermal equilibrium, electromagnetic modes, spectral energy density, and asymptotic approximation.

## Objectives

1. Evaluate Planck's wavelength-domain spectral energy density with explicit units.
2. Compare the exact density with the short-wavelength Wien approximation and long-wavelength Rayleigh--Jeans approximation.
3. Verify each approximation only in its corresponding dimensionless regime.
4. Plot exact spectra across temperature and compare the limiting formulas.

The represented quantity is energy per volume per wavelength. It is not spectral radiance or emitted surface flux.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.thermal.statmech.radiation.blackbody import (
    BlackbodyWavelengthSpectrumModel,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

## Mathematical model

For wavelength $\lambda>0$ and absolute temperature $T>0$, the equilibrium spectral energy density per wavelength is

$$
u_\lambda(\lambda,T)=
\frac{8\pi hc}{\lambda^5}
\frac{1}{\exp\!\left(hc/(\lambda k_BT)\right)-1}.
$$

Writing $x=hc/(\lambda k_BT)$, the Wien approximation applies for $x\gg1$ and the Rayleigh--Jeans approximation applies for $x\ll1$.


In [ ]:
minimum_wavelength_nanometer = 100.0
maximum_wavelength_nanometer = 100_000.0
wavelength_sample_count = 1200
temperature_values_kelvin = (3000.0, 4000.0, 5000.0)
wavelength_unit = PhysicalUnit(expression="nanometer")
spectral_density_unit = PhysicalUnit(expression="joule / meter ** 4")

wavelength_values_nanometer = np.geomspace(
    minimum_wavelength_nanometer,
    maximum_wavelength_nanometer,
    wavelength_sample_count,
).astype(np.float64)
wavelengths = VectorQuantity(
    magnitude=wavelength_values_nanometer,
    unit=wavelength_unit,
)
model = BlackbodyWavelengthSpectrumModel()
evaluations = tuple(
    model.evaluate(
        wavelengths=wavelengths,
        temperature=ScalarQuantity(
            magnitude=temperature_kelvin,
            unit=PhysicalUnit(expression="kelvin"),
        ),
        spectral_energy_density_unit=spectral_density_unit,
    )
    for temperature_kelvin in temperature_values_kelvin
)

In [ ]:
reference_evaluation = evaluations[-1]
wavelength_values_metres = wavelength_values_nanometer * 1.0e-9
reference_temperature_kelvin = reference_evaluation.request.temperature.magnitude
exponent = (
    SI.h * SI.c / (wavelength_values_metres * SI.k_B * reference_temperature_kelvin)
)
short_wavelength_threshold = 8.0
long_wavelength_threshold = 0.1
short_wavelength_mask = exponent >= short_wavelength_threshold
long_wavelength_mask = exponent <= long_wavelength_threshold
wien_relative_tolerance = 4.0e-4
rayleigh_jeans_relative_tolerance = 5.2e-2
assert np.allclose(
    reference_evaluation.wien_spectral_energy_density.magnitude[short_wavelength_mask],
    reference_evaluation.planck_spectral_energy_density.magnitude[
        short_wavelength_mask
    ],
    rtol=wien_relative_tolerance,
    atol=0.0,
)
assert np.allclose(
    reference_evaluation.rayleigh_jeans_spectral_energy_density.magnitude[
        long_wavelength_mask
    ],
    reference_evaluation.planck_spectral_energy_density.magnitude[long_wavelength_mask],
    rtol=rayleigh_jeans_relative_tolerance,
    atol=0.0,
)

In [ ]:
figure_size_inches = (12.0, 4.8)
figure, (temperature_axis, approximation_axis) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=figure_size_inches,
)
for evaluation in evaluations:
    temperature_axis.loglog(
        evaluation.request.wavelengths.magnitude,
        evaluation.planck_spectral_energy_density.magnitude,
        label=f"{evaluation.request.temperature.magnitude:.0f} K",
    )
temperature_axis.set(
    xlabel="wavelength (nm)",
    ylabel="spectral energy density (J/m⁴)",
    title="Planck spectra",
)
temperature_axis.grid(True, which="both")
temperature_axis.legend(title="temperature")

approximation_axis.loglog(
    wavelength_values_nanometer,
    reference_evaluation.planck_spectral_energy_density.magnitude,
    label="Planck",
)
approximation_axis.loglog(
    wavelength_values_nanometer,
    reference_evaluation.wien_spectral_energy_density.magnitude,
    linestyle="--",
    label="Wien approximation",
)
approximation_axis.loglog(
    wavelength_values_nanometer,
    reference_evaluation.rayleigh_jeans_spectral_energy_density.magnitude,
    linestyle=":",
    label="Rayleigh–Jeans approximation",
)
approximation_axis.set(
    xlabel="wavelength (nm)",
    ylabel="spectral energy density (J/m⁴)",
    title=f"Approximations at {reference_temperature_kelvin:.0f} K",
)
approximation_axis.grid(True, which="both")
approximation_axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

Increasing temperature raises the spectral energy density and moves its wavelength-domain maximum toward shorter wavelength. The Wien curve approaches Planck's law at large $x$, while the Rayleigh--Jeans curve approaches it at small $x$. Outside those regimes the limiting formulas should not be interpreted as accurate substitutes.

These ideal equilibrium curves are illustrative software and numerical-verification results. They do not validate emissivity or radiative behavior for a real material.

## Exercises

1. Derive both asymptotic formulas from Planck's law using $x\gg1$ and $x\ll1$.
2. Locate each numerical Planck maximum and investigate its inverse-temperature scaling.
3. Convert the wavelength coordinate to micrometres and verify that the represented physical spectra remain unchanged.
4. Explain the additional angular and geometric factors needed to convert energy density into spectral radiance or surface flux.
